# Sistema de Recomendação de Transporte Público

O programa recebe uma origem e um destino, procura uma viagem direta na base GTFS e consulta a API Olho Vivo para confirmar a linha e buscar informações em tempo real quando a SPTrans disponibiliza os dados.

A busca considera o raio configurado, a sequência dos pontos na mesma viagem e o traçado do `shapes.txt`.


## 1. Bibliotecas e conexão com a SPTrans

O token fica somente no arquivo `.env`, fora do código e do GitHub.

In [ ]:
# -*- coding: utf-8 -*-
import os
import math
import zipfile
from pathlib import Path
from datetime import datetime, date
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
import folium
from dotenv import load_dotenv
from geopy.geocoders import Nominatim
from IPython.display import display, HTML, Markdown

try:
    from ipyleaflet import Map, Marker, AwesomeIcon
    import ipywidgets as widgets
    MAPA_CLICAVEL_DISPONIVEL = True
except ImportError:
    MAPA_CLICAVEL_DISPONIVEL = False


def encontrar_pasta_projeto():
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "dados" / "gtfs_sptrans.zip").exists():
            return pasta
    raise FileNotFoundError("Não encontrei dados/gtfs_sptrans.zip. Abra o notebook pela pasta principal do projeto.")


PASTA_PROJETO = encontrar_pasta_projeto()
PASTA_MAPAS = PASTA_PROJETO / "mapas"
PASTA_MAPAS.mkdir(exist_ok=True)
load_dotenv(PASTA_PROJETO / ".env", override=True)

URL_API = "https://api.olhovivo.sptrans.com.br/v2.1"
TOKEN_SPTRANS = os.getenv("SPTRANS_TOKEN", "").strip()
RAIO_BUSCA_METROS = 500


class OlhoVivo:
    def __init__(self, token):
        self.token = token
        self.sessao = requests.Session()
        self.autenticado = False

    def autenticar(self):
        if not self.token:
            raise ValueError("Token ausente. Configure SPTRANS_TOKEN no arquivo .env.")
        resposta = self.sessao.post(
            f"{URL_API}/Login/Autenticar",
            params={"token": self.token},
            timeout=20
        )
        resposta.raise_for_status()
        try:
            autenticou = resposta.json()
        except ValueError as erro:
            raise RuntimeError("A SPTrans retornou uma resposta inválida na autenticação.") from erro
        if autenticou is not True:
            raise PermissionError("A SPTrans não aceitou o token. Confira o cadastro e o arquivo .env.")
        self.autenticado = True
        return True

    def consultar(self, endpoint, parametros=None):
        if not self.autenticado:
            raise RuntimeError("A API ainda não foi autenticada.")
        resposta = self.sessao.get(
            f"{URL_API}/{endpoint.lstrip('/')}",
            params=parametros or {},
            timeout=25
        )
        resposta.raise_for_status()
        try:
            return resposta.json()
        except ValueError as erro:
            raise RuntimeError(f"A resposta de {endpoint} não é um JSON válido.") from erro


api = None
if TOKEN_SPTRANS:
    api = OlhoVivo(TOKEN_SPTRANS)
    try:
        api.autenticar()
        print("Conexão com a API Olho Vivo: autenticada.")
    except Exception as erro:
        api = None
        print(f"Não foi possível autenticar na API: {erro}")
        print("A análise GTFS ainda pode ser executada, mas não haverá previsão em tempo real.")
else:
    print("Token não configurado. Crie o arquivo .env a partir do .env.example.")


## 2. Carregar a base GTFS

Os arquivos são lidos diretamente do ZIP que acompanha o projeto. Os identificadores entre linhas, viagens, paradas e formas são conferidos antes da busca.

In [ ]:
ARQUIVOS_GTFS = {
    "routes": ["route_id", "route_short_name", "route_long_name", "route_type"],
    "trips": ["route_id", "service_id", "trip_id", "trip_headsign", "direction_id", "shape_id"],
    "stops": ["stop_id", "stop_name", "stop_lat", "stop_lon"],
    "stop_times": ["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
    "shapes": ["shape_id", "shape_pt_lat", "shape_pt_lon", "shape_pt_sequence"],
    "calendar": ["service_id", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday", "start_date", "end_date"]
}


def carregar_gtfs():
    arquivo = PASTA_PROJETO / "dados" / "gtfs_sptrans.zip"
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo GTFS não encontrado: {arquivo}")
    dados = {}
    with zipfile.ZipFile(arquivo) as pacote:
        nomes = set(pacote.namelist())
        for nome_opcional in ["agency.txt", "fare_attributes.txt", "fare_rules.txt", "frequencies.txt"]:
            if nome_opcional in nomes:
                chave = nome_opcional.replace(".txt", "")
                dados[chave] = pd.read_csv(pacote.open(nome_opcional), dtype=str, low_memory=False)
        for tabela, colunas in ARQUIVOS_GTFS.items():
            nome = f"{tabela}.txt"
            if nome not in nomes:
                raise FileNotFoundError(f"O arquivo {nome} não está dentro do GTFS.")
            dados[tabela] = pd.read_csv(pacote.open(nome), dtype=str, low_memory=False)
            faltantes = [coluna for coluna in colunas if coluna not in dados[tabela].columns]
            if faltantes:
                raise ValueError(f"Colunas ausentes em {nome}: {faltantes}")
        if "calendar_dates.txt" in nomes:
            dados["calendar_dates"] = pd.read_csv(pacote.open("calendar_dates.txt"), dtype=str)
        else:
            dados["calendar_dates"] = pd.DataFrame(columns=["service_id", "date", "exception_type"])
    dados["stops"]["stop_lat"] = pd.to_numeric(dados["stops"]["stop_lat"], errors="coerce")
    dados["stops"]["stop_lon"] = pd.to_numeric(dados["stops"]["stop_lon"], errors="coerce")
    dados["stop_times"]["stop_sequence"] = pd.to_numeric(dados["stop_times"]["stop_sequence"], errors="coerce")
    dados["shapes"]["shape_pt_sequence"] = pd.to_numeric(dados["shapes"]["shape_pt_sequence"], errors="coerce")
    dados["shapes"]["shape_pt_lat"] = pd.to_numeric(dados["shapes"]["shape_pt_lat"], errors="coerce")
    dados["shapes"]["shape_pt_lon"] = pd.to_numeric(dados["shapes"]["shape_pt_lon"], errors="coerce")

    # Confere as ligações entre os arquivos antes de iniciar a busca.
    if not set(dados["trips"]["route_id"]).issubset(set(dados["routes"]["route_id"])):
        raise ValueError("Existem viagens com route_id sem correspondência em routes.txt.")
    if not set(dados["stop_times"]["trip_id"]).issubset(set(dados["trips"]["trip_id"])):
        raise ValueError("Existem horários com trip_id sem correspondência em trips.txt.")
    if not set(dados["stop_times"]["stop_id"]).issubset(set(dados["stops"]["stop_id"])):
        raise ValueError("Existem horários com stop_id sem correspondência em stops.txt.")
    if not set(dados["trips"]["shape_id"]).issubset(set(dados["shapes"]["shape_id"])):
        raise ValueError("Existem viagens com shape_id sem correspondência em shapes.txt.")
    if not set(dados["trips"]["service_id"]).issubset(set(dados["calendar"]["service_id"])):
        raise ValueError("Existem viagens com service_id sem correspondência em calendar.txt.")
    return dados


gtfs = carregar_gtfs()
print(f"GTFS carregado: {len(gtfs['routes']):,} linhas, {len(gtfs['stops']):,} paradas e {len(gtfs['trips']):,} viagens.")


## 3. Distâncias e viagens diretas

A distância é calculada pela fórmula de Haversine. Uma opção só é considerada direta quando o embarque e o desembarque estão na mesma viagem e na ordem correta.

In [ ]:
def servicos_ativos(dados, dia=None):
    if dia is None:
        dia = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
    calendario = dados["calendar"].copy()
    data_num = int(dia.strftime("%Y%m%d"))
    coluna_dia = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"][dia.weekday()]
    calendario["start_date"] = pd.to_numeric(calendario["start_date"], errors="coerce")
    calendario["end_date"] = pd.to_numeric(calendario["end_date"], errors="coerce")
    ativos = set(calendario.loc[
        (calendario[coluna_dia] == "1") &
        (calendario["start_date"] <= data_num) &
        (calendario["end_date"] >= data_num), "service_id"
    ])
    excecoes = dados.get("calendar_dates", pd.DataFrame())
    if not excecoes.empty:
        hoje = excecoes[excecoes["date"].astype(str) == str(data_num)]
        ativos.update(hoje.loc[hoje["exception_type"].astype(str) == "1", "service_id"])
        ativos.difference_update(hoje.loc[hoje["exception_type"].astype(str) == "2", "service_id"])
    return ativos


def distancia_metros(lat1, lon1, lat2, lon2):
    raio = 6371000
    lat1, lon1, lat2, lon2 = map(math.radians, [float(lat1), float(lon1), float(lat2), float(lon2)])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 2 * raio * math.asin(math.sqrt(a))


def validar_coordenada(ponto):
    if not isinstance(ponto, (tuple, list)) or len(ponto) != 2:
        raise ValueError("Informe latitude e longitude.")
    lat, lon = float(ponto[0]), float(ponto[1])
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        raise ValueError("Latitude deve estar entre -90 e 90 e longitude entre -180 e 180.")
    return lat, lon


def paradas_proximas(dados, ponto, raio=500):
    lat, lon = validar_coordenada(ponto)
    if not 1 <= int(raio) <= 1000:
        raise ValueError("O raio deve estar entre 1 e 1.000 metros.")
    paradas = dados["stops"].dropna(subset=["stop_lat", "stop_lon"]).copy()
    if (lat < paradas["stop_lat"].min() - 0.20 or lat > paradas["stop_lat"].max() + 0.20 or
        lon < paradas["stop_lon"].min() - 0.20 or lon > paradas["stop_lon"].max() + 0.20):
        raise ValueError("A coordenada está fora da área coberta pela base GTFS da SPTrans.")
    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    lat2 = np.radians(paradas["stop_lat"].to_numpy(dtype=float))
    lon2 = np.radians(paradas["stop_lon"].to_numpy(dtype=float))
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    paradas["distancia_m"] = 2 * 6371000 * np.arcsin(np.sqrt(a))
    return paradas[paradas["distancia_m"] <= int(raio)].sort_values("distancia_m")


def buscar_viagens_diretas(dados, origem, destino, raio=500, dia=None):
    paradas_o = paradas_proximas(dados, origem, raio)
    paradas_d = paradas_proximas(dados, destino, raio)
    if paradas_o.empty or paradas_d.empty:
        return pd.DataFrame(), paradas_o, paradas_d

    tempos = dados["stop_times"].dropna(subset=["stop_sequence"]).copy()
    origens = tempos[tempos["stop_id"].isin(paradas_o["stop_id"])][["trip_id", "stop_id", "stop_sequence"]]
    destinos = tempos[tempos["stop_id"].isin(paradas_d["stop_id"])][["trip_id", "stop_id", "stop_sequence"]]
    origens = origens.rename(columns={"stop_id": "embarque_id", "stop_sequence": "seq_embarque"})
    destinos = destinos.rename(columns={"stop_id": "desembarque_id", "stop_sequence": "seq_desembarque"})
    pares = origens.merge(destinos, on="trip_id")
    pares = pares[pares["seq_embarque"] < pares["seq_desembarque"]]
    if pares.empty:
        return pd.DataFrame(), paradas_o, paradas_d

    ativos = servicos_ativos(dados, dia)
    viagens = dados["trips"][dados["trips"]["service_id"].isin(ativos)].copy()
    pares = pares.merge(viagens, on="trip_id", how="inner")
    rotas = dados["routes"].copy()
    pares = pares.merge(rotas, on="route_id", how="inner")
    pares = pares[pares["route_type"].astype(str) == "3"].copy()
    pares = pares.merge(
        paradas_o[["stop_id", "stop_name", "distancia_m"]],
        left_on="embarque_id", right_on="stop_id", how="left"
    ).drop(columns="stop_id")
    pares = pares.rename(columns={"stop_name": "nome_embarque", "distancia_m": "distancia_origem_m"})
    pares = pares.merge(
        paradas_d[["stop_id", "stop_name", "distancia_m"]],
        left_on="desembarque_id", right_on="stop_id", how="left"
    ).drop(columns="stop_id")
    pares = pares.rename(columns={"stop_name": "nome_desembarque", "distancia_m": "distancia_destino_m"})
    pares["quantidade_paradas"] = pares["seq_desembarque"] - pares["seq_embarque"] + 1
    pares["caminhada_total_m"] = pares["distancia_origem_m"] + pares["distancia_destino_m"]
    pares = pares.sort_values(["caminhada_total_m", "quantidade_paradas"])
    return pares.reset_index(drop=True), paradas_o, paradas_d


def obter_paradas_viagem(dados, trip_id, seq_inicio, seq_fim):
    tempos = dados["stop_times"]
    trecho = tempos[(tempos["trip_id"] == trip_id) &
                    (tempos["stop_sequence"] >= seq_inicio) &
                    (tempos["stop_sequence"] <= seq_fim)].copy()
    trecho = trecho.sort_values("stop_sequence")
    return trecho.merge(dados["stops"], on="stop_id", how="left")


def obter_trecho_shape(dados, shape_id, ponto_embarque, ponto_desembarque, tolerancia=250):
    forma = dados["shapes"][dados["shapes"]["shape_id"].astype(str) == str(shape_id)].copy()
    forma = forma.dropna(subset=["shape_pt_lat", "shape_pt_lon"]).sort_values("shape_pt_sequence")
    if len(forma) < 2:
        return None
    dist_embarque = forma.apply(lambda r: distancia_metros(*ponto_embarque, r["shape_pt_lat"], r["shape_pt_lon"]), axis=1)
    dist_desembarque = forma.apply(lambda r: distancia_metros(*ponto_desembarque, r["shape_pt_lat"], r["shape_pt_lon"]), axis=1)
    idx_embarque = int(np.argmin(dist_embarque.to_numpy()))
    idx_desembarque = int(np.argmin(dist_desembarque.to_numpy()))
    if dist_embarque.iloc[idx_embarque] > tolerancia or dist_desembarque.iloc[idx_desembarque] > tolerancia:
        return None
    if idx_embarque >= idx_desembarque:
        return None
    trecho = forma.iloc[idx_embarque:idx_desembarque + 1]
    return list(zip(trecho["shape_pt_lat"].astype(float), trecho["shape_pt_lon"].astype(float)))


def geocodificar(endereco):
    if not str(endereco).strip():
        raise ValueError("Digite um endereço para pesquisar.")
    geolocalizador = Nominatim(user_agent="sistema_recomendacao_transporte_publico")
    resultado = geolocalizador.geocode(f"{endereco}, São Paulo, SP, Brasil", timeout=15)
    if resultado is None:
        raise ValueError(f"Endereço não encontrado: {endereco}")
    return float(resultado.latitude), float(resultado.longitude)


## 4. Consultas complementares da API

A linha encontrada no GTFS é procurada na Olho Vivo. A previsão só é consultada quando a API disponibiliza um código oficial de parada compatível com o ponto de embarque.

In [ ]:
def normalizar_linha(valor):
    return str(valor).upper().replace(" ", "").replace("–", "-")


def buscar_linha_na_api(cliente, numero_linha):
    numero_busca = str(numero_linha).split("-")[0]
    resultados = cliente.consultar("Linha/Buscar", {"termosBusca": numero_busca})
    if not isinstance(resultados, list):
        return None
    alvo = normalizar_linha(numero_linha)
    for linha in resultados:
        try:
            letreiro = f"{linha.get('lt', '')}-{int(linha.get('tl'))}"
        except (TypeError, ValueError):
            continue
        if normalizar_linha(letreiro) == alvo:
            return linha
    return None


def normalizar_parada_api(parada):
    return {
        "codigo": parada.get("cp", parada.get("CodigoParada")),
        "nome": parada.get("np", parada.get("Nome", "Parada")),
        "latitude": parada.get("py", parada.get("Latitude")),
        "longitude": parada.get("px", parada.get("Longitude"))
    }


def localizar_parada_api(cliente, linha_api, parada_gtfs):
    codigo_linha = linha_api.get("cl")
    respostas = []
    try:
        respostas = cliente.consultar("Parada/BuscarParadasPorLinha", {"codigoLinha": codigo_linha}) or []
    except Exception:
        respostas = []
    nome = str(parada_gtfs["stop_name"])
    try:
        respostas_nome = cliente.consultar("Parada/Buscar", {"termosBusca": nome}) or []
        respostas.extend(respostas_nome)
    except Exception:
        pass
    candidatos = []
    vistos = set()
    for item in respostas:
        p = normalizar_parada_api(item)
        if p["codigo"] is None or p["latitude"] is None or p["longitude"] is None:
            continue
        if str(p["codigo"]) in vistos:
            continue
        vistos.add(str(p["codigo"]))
        p["distancia_m"] = distancia_metros(
            parada_gtfs["stop_lat"], parada_gtfs["stop_lon"], p["latitude"], p["longitude"]
        )
        candidatos.append(p)
    if not candidatos:
        return None
    melhor = min(candidatos, key=lambda x: x["distancia_m"])
    return melhor if melhor["distancia_m"] <= 150 else None


def consultar_previsao(cliente, codigo_linha, codigo_parada):
    dados = cliente.consultar("Previsao", {
        "codigoParada": int(codigo_parada),
        "codigoLinha": int(codigo_linha)
    })
    referencia = dados.get("hr") if isinstance(dados, dict) else None
    ponto = dados.get("p", {}) if isinstance(dados, dict) else {}
    linhas = ponto.get("l", []) if isinstance(ponto, dict) else []
    veiculos = []
    for linha in linhas:
        if str(linha.get("cl")) != str(codigo_linha):
            continue
        veiculos.extend(linha.get("vs", []) or [])
    veiculos = [v for v in veiculos if v.get("t")]
    veiculos.sort(key=lambda v: str(v.get("t")))
    return referencia, veiculos


def consultar_posicoes(cliente, codigo_linha):
    dados = cliente.consultar("Posicao/Linha", {"codigoLinha": int(codigo_linha)})
    if not isinstance(dados, dict):
        return None, []
    return dados.get("hr"), dados.get("vs", [])


## 5. Escolher origem e destino no mapa

Selecione **Origem** no botão e clique no mapa. Depois selecione **Destino** e clique no segundo local. Também é possível usar endereço ou coordenadas nas configurações abaixo.

In [ ]:
# Selecione o tipo de ponto e clique no mapa: primeiro escolha a origem, depois o destino.
pontos_escolhidos = {"origem": None, "destino": None}

if MAPA_CLICAVEL_DISPONIVEL:
    centro = (
        float(gtfs["stops"]["stop_lat"].mean()),
        float(gtfs["stops"]["stop_lon"].mean())
    )
    mapa_selecao = Map(center=centro, zoom=11, scroll_wheel_zoom=True)
    tipo_ponto = widgets.ToggleButtons(
        options=[("Origem", "origem"), ("Destino", "destino")],
        value="origem", description="Selecionar:"
    )
    informacao_pontos = widgets.HTML(value="Clique no mapa para marcar a origem.")
    marcadores = {}

    def registrar_clique(**kwargs):
        if kwargs.get("type") != "click":
            return
        latitude, longitude = kwargs.get("coordinates", (None, None))
        if latitude is None or longitude is None:
            return
        tipo = tipo_ponto.value
        pontos_escolhidos[tipo] = (float(latitude), float(longitude))
        if tipo in marcadores:
            mapa_selecao.remove_layer(marcadores[tipo])
        cor = "green" if tipo == "origem" else "red"
        icone = "home" if tipo == "origem" else "flag-checkered"
        marcador = Marker(
            location=(latitude, longitude),
            title="Origem" if tipo == "origem" else "Destino",
            icon=AwesomeIcon(name=icone, marker_color=cor, icon_color="white")
        )
        mapa_selecao.add_layer(marcador)
        marcadores[tipo] = marcador
        informacao_pontos.value = (
            f"<b>Origem:</b> {pontos_escolhidos['origem']}<br>"
            f"<b>Destino:</b> {pontos_escolhidos['destino']}"
        )

    mapa_selecao.on_interaction(registrar_clique)
    display(Markdown("### Escolha a origem e o destino no mapa"))
    display(widgets.VBox([tipo_ponto, mapa_selecao, informacao_pontos]))
else:
    print("Para selecionar pontos no mapa, instale as dependências e reinicie o kernel: pip install -r requirements.txt")


## 6. Configurar a consulta e montar o mapa

O raio inicial é de 500 metros e pode ser alterado entre 1 e 1.000 metros. O mapa final usa o trecho correspondente do `shapes.txt`; se não houver forma válida na direção correta, o programa informa a limitação em vez de desenhar uma linha fictícia.

In [ ]:
# Escolha como informar os locais: mapa, endereço ou coordenadas.
MODO_ENTRADA = "mapa"  # use: mapa, endereco ou coordenadas
ENDERECO_ORIGEM = "Avenida Paulista, São Paulo"
ENDERECO_DESTINO = "Parque Ibirapuera, São Paulo"
COORDENADA_ORIGEM = None  # exemplo: (-23.5614, -46.6559)
COORDENADA_DESTINO = None  # exemplo: (-23.5874, -46.6576)


def obter_origem_destino():
    if MODO_ENTRADA == "mapa":
        origem = pontos_escolhidos["origem"]
        destino = pontos_escolhidos["destino"]
        if origem is None or destino is None:
            raise ValueError("Marque a origem e o destino no mapa antes de executar esta etapa.")
    elif MODO_ENTRADA == "endereco":
        origem = geocodificar(ENDERECO_ORIGEM)
        destino = geocodificar(ENDERECO_DESTINO)
    elif MODO_ENTRADA == "coordenadas":
        origem = validar_coordenada(COORDENADA_ORIGEM)
        destino = validar_coordenada(COORDENADA_DESTINO)
    else:
        raise ValueError("MODO_ENTRADA deve ser mapa, endereco ou coordenadas.")
    return validar_coordenada(origem), validar_coordenada(destino)


def gerar_mapa(dados, candidato, origem, destino, paradas_trecho, forma, linha_api=None,
               previsoes=None, posicoes=None, caminho_html=None):
    centro = [(origem[0] + destino[0]) / 2, (origem[1] + destino[1]) / 2]
    mapa = folium.Map(location=centro, zoom_start=13, control_scale=True, tiles="OpenStreetMap")
    folium.Marker(origem, tooltip="Origem", popup="Origem informada", icon=folium.Icon(color="green", icon="home")).add_to(mapa)
    folium.Marker(destino, tooltip="Destino", popup="Destino informado", icon=folium.Icon(color="red", icon="flag")).add_to(mapa)

    embarque = dados["stops"].loc[dados["stops"]["stop_id"] == candidato["embarque_id"]].iloc[0]
    desembarque = dados["stops"].loc[dados["stops"]["stop_id"] == candidato["desembarque_id"]].iloc[0]
    folium.Marker(
        (embarque.stop_lat, embarque.stop_lon), tooltip="Embarque",
        popup=f"Embarque: {embarque.stop_name}", icon=folium.Icon(color="blue", icon="play")
    ).add_to(mapa)
    folium.Marker(
        (desembarque.stop_lat, desembarque.stop_lon), tooltip="Desembarque",
        popup=f"Desembarque: {desembarque.stop_name}", icon=folium.Icon(color="purple", icon="stop")
    ).add_to(mapa)

    pontos_limites = [origem, destino, (embarque.stop_lat, embarque.stop_lon), (desembarque.stop_lat, desembarque.stop_lon)]
    for _, parada in paradas_trecho.iterrows():
        if str(parada["stop_id"]) in {str(candidato["embarque_id"]), str(candidato["desembarque_id"])}:
            continue
        coordenada = (float(parada["stop_lat"]), float(parada["stop_lon"]))
        folium.CircleMarker(
            coordenada, radius=4, color="#555555", fill=True, fill_opacity=0.85,
            tooltip=f"Parada intermediária: {parada['stop_name']}"
        ).add_to(mapa)
        pontos_limites.append(coordenada)

    if forma:
        folium.PolyLine(forma, color="#1769aa", weight=5, opacity=0.85,
                        tooltip=f"Trecho da linha {candidato['route_short_name']} pelo GTFS").add_to(mapa)
        pontos_limites.extend(forma)

    for veiculo in posicoes or []:
        if veiculo.get("py") is None or veiculo.get("px") is None:
            continue
        coordenada = (float(veiculo["py"]), float(veiculo["px"]))
        folium.Marker(
            coordenada, tooltip=f"Ônibus {veiculo.get('p', 'sem prefixo')}",
            popup=f"Prefixo: {veiculo.get('p', 'não informado')}<br>Acessível: {veiculo.get('a', 'não informado')}",
            icon=folium.Icon(color="orange", icon="bus", prefix="fa")
        ).add_to(mapa)
        pontos_limites.append(coordenada)

    legenda = """
    <div style='position: fixed; bottom: 25px; left: 25px; z-index: 9999; background: white;
                padding: 10px 12px; border: 1px solid #888; border-radius: 5px; font-size: 13px;'>
      <b>Legenda</b><br>
      <span style='color:green'>●</span> Origem<br>
      <span style='color:red'>●</span> Destino<br>
      <span style='color:blue'>●</span> Embarque<br>
      <span style='color:purple'>●</span> Desembarque<br>
      <span style='color:#555'>●</span> Paradas intermediárias<br>
      <span style='color:orange'>●</span> Ônibus em circulação<br>
      <span style='color:#1769aa'>━</span> Traçado GTFS
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(legenda))
    mapa.fit_bounds(pontos_limites)
    if caminho_html:
        mapa.save(str(caminho_html))
    return mapa


## 7. Executar a recomendação

Depois de marcar os dois locais no mapa, execute esta célula. Ela mostra a linha, sentido, embarque, desembarque, paradas intermediárias, previsão (quando disponível) e o mapa interativo.

In [ ]:
# Execute esta célula depois de marcar os dois pontos no mapa.
origem, destino = obter_origem_destino()
if not 1 <= RAIO_BUSCA_METROS <= 1000:
    raise ValueError("RAIO_BUSCA_METROS precisa ficar entre 1 e 1.000 metros.")

candidatos, paradas_origem, paradas_destino = buscar_viagens_diretas(
    gtfs, origem, destino, RAIO_BUSCA_METROS
)

if candidatos.empty:
    if paradas_origem.empty:
        print("Nenhuma parada foi encontrada perto da origem dentro do raio informado.")
    elif paradas_destino.empty:
        print("Nenhuma parada foi encontrada perto do destino dentro do raio informado.")
    else:
        print("Nenhuma viagem direta foi encontrada para esse par de locais e raio.")
else:
    print(f"Paradas próximas da origem: {len(paradas_origem)}")
    print(f"Paradas próximas do destino: {len(paradas_destino)}")
    selecionado = None
    linha_confirmada = None

    if api is not None:
        for _, opcao in candidatos.drop_duplicates("route_short_name").head(20).iterrows():
            try:
                linha = buscar_linha_na_api(api, opcao["route_short_name"])
                if linha:
                    selecionado = opcao.copy()
                    linha_confirmada = linha
                    break
            except (requests.ConnectionError, requests.Timeout) as erro:
                print(f"Sem comunicação com a SPTrans neste momento: {erro}")
                break
            except Exception as erro:
                print(f"Falha ao consultar a linha {opcao['route_short_name']}: {erro}")
    if selecionado is None:
        selecionado = candidatos.iloc[0].copy()
        print("Atenção: a opção abaixo veio do GTFS e não foi confirmada pela API ao vivo.")

    paradas_trecho = obter_paradas_viagem(
        gtfs, selecionado["trip_id"], selecionado["seq_embarque"], selecionado["seq_desembarque"]
    )
    parada_embarque = gtfs["stops"].loc[gtfs["stops"]["stop_id"] == selecionado["embarque_id"]].iloc[0]
    parada_desembarque = gtfs["stops"].loc[gtfs["stops"]["stop_id"] == selecionado["desembarque_id"]].iloc[0]
    forma = obter_trecho_shape(
        gtfs, selecionado["shape_id"],
        (parada_embarque["stop_lat"], parada_embarque["stop_lon"]),
        (parada_desembarque["stop_lat"], parada_desembarque["stop_lon"])
    )

    previsoes = []
    posicoes = []
    referencia_previsao = None
    parada_api = None
    if linha_confirmada:
        try:
            parada_api = localizar_parada_api(api, linha_confirmada, parada_embarque)
            if parada_api:
                referencia_previsao, previsoes = consultar_previsao(
                    api, linha_confirmada["cl"], parada_api["codigo"]
                )
            else:
                print("A parada de embarque não foi localizada entre as paradas publicadas pela API; previsão indisponível.")
        except Exception as erro:
            print(f"Não foi possível consultar a previsão: {erro}")
        try:
            _, posicoes = consultar_posicoes(api, linha_confirmada["cl"])
        except Exception as erro:
            print(f"Não foi possível consultar a posição dos veículos: {erro}")

    display(Markdown(f"## Linha selecionada: {selecionado['route_short_name']}"))
    if linha_confirmada:
        sentido_api = f"{linha_confirmada.get('sl')} — {linha_confirmada.get('tp') if linha_confirmada.get('sl') == 1 else linha_confirmada.get('ts')}"
    else:
        sentido_api = "não confirmado pela API"
    print(f"Destino da viagem no GTFS: {selecionado.get('trip_headsign', 'não informado')} | Sentido SPTrans: {sentido_api}")
    print(f"Embarque: {parada_embarque['stop_name']} — {selecionado['distancia_origem_m']:.0f} m da origem")
    print(f"Desembarque: {parada_desembarque['stop_name']} — {selecionado['distancia_destino_m']:.0f} m do destino")
    print(f"Caminhada total estimada em linha reta: {selecionado['caminhada_total_m']:.0f} m")
    print(f"Quantidade de paradas no trecho (incluindo embarque e desembarque): {len(paradas_trecho)}")

    if previsoes:
        print(f"Previsão de chegada (referência {referencia_previsao}):")
        for veiculo in previsoes:
            print(f"  Ônibus {veiculo.get('p', 'não informado')} — chegada prevista {veiculo.get('t')} — acessível: {veiculo.get('a', 'não informado')}")
    else:
        print("Previsão de chegada indisponível no momento.")

    if not forma:
        print("O shape da viagem não permitiu identificar o trecho na direção correta; o mapa não desenhará uma linha fictícia.")
    if linha_confirmada and posicoes:
        print(f"Veículos localizados pela API: {len(posicoes)}")

    mapa_resultado = gerar_mapa(
        gtfs, selecionado, origem, destino, paradas_trecho, forma,
        linha_api=linha_confirmada, previsoes=previsoes, posicoes=posicoes,
        caminho_html=PASTA_MAPAS / "rota_recomendada.html"
    )
    display(mapa_resultado)
    print(f"Mapa salvo em: {PASTA_MAPAS / 'rota_recomendada.html'}")
    display(paradas_trecho[["stop_sequence", "stop_name", "stop_lat", "stop_lon"]].reset_index(drop=True))


## 8. Testes locais

Estes testes verificam cálculos e leitura do GTFS sem depender da internet nem do token. A autenticação e as consultas em tempo real são testadas quando o notebook é executado com um token válido.

In [ ]:
# Testes locais: não consultam a internet nem usam token.
def executar_testes_locais():
    assert 110000 <= distancia_metros(0, 0, 0, 1) <= 112000
    assert validar_coordenada((-23.55, -46.63)) == (-23.55, -46.63)

    try:
        validar_coordenada((95, 0))
        raise AssertionError("Coordenada inválida foi aceita.")
    except ValueError:
        pass

    origem_teste = (-23.520349, -46.704819)
    destino_teste = (-23.525799, -46.679251)
    dia_teste = date(2026, 11, 26)
    paradas_o = paradas_proximas(gtfs, origem_teste, 500)
    paradas_d = paradas_proximas(gtfs, destino_teste, 500)
    assert not paradas_o.empty and not paradas_d.empty

    opcoes, _, _ = buscar_viagens_diretas(
        gtfs, origem_teste, destino_teste, 500, dia=dia_teste
    )
    assert not opcoes.empty, "Nenhuma viagem direta foi encontrada no conjunto de teste."
    assert (opcoes["seq_embarque"] < opcoes["seq_desembarque"]).all()

    melhor = opcoes.iloc[0]
    paradas = obter_paradas_viagem(
        gtfs, melhor["trip_id"], melhor["seq_embarque"], melhor["seq_desembarque"]
    )
    assert len(paradas) >= 2

    embarque = gtfs["stops"].loc[gtfs["stops"]["stop_id"] == melhor["embarque_id"]].iloc[0]
    desembarque = gtfs["stops"].loc[gtfs["stops"]["stop_id"] == melhor["desembarque_id"]].iloc[0]
    forma = obter_trecho_shape(
        gtfs, melhor["shape_id"],
        (embarque["stop_lat"], embarque["stop_lon"]),
        (desembarque["stop_lat"], desembarque["stop_lon"])
    )
    assert forma is not None and len(forma) >= 2

    class APIFalsa:
        def consultar(self, endpoint, parametros=None):
            if endpoint == "Linha/Buscar":
                return [{"cl": 123, "lt": "877T", "tl": 10, "sl": 1, "tp": "Terminal A", "ts": "Terminal B"}]
            if endpoint == "Previsao":
                return {"hr": "10:00", "p": {"l": [{"cl": 123, "vs": [{"p": "12345", "t": "10:12", "a": True}]}]}}
            if endpoint == "Posicao/Linha":
                return {"hr": "10:00", "vs": [{"p": "12345", "py": -23.5, "px": -46.6}]}
            raise ValueError("Endpoint inesperado no teste")

    api_falsa = APIFalsa()
    assert buscar_linha_na_api(api_falsa, "877T-10")["cl"] == 123
    referencia, previsoes_teste = consultar_previsao(api_falsa, 123, 456)
    assert referencia == "10:00" and previsoes_teste[0]["t"] == "10:12"
    hora_posicao, veiculos_teste = consultar_posicoes(api_falsa, 123)
    assert hora_posicao == "10:00" and veiculos_teste[0]["p"] == "12345"

    try:
        OlhoVivo("").autenticar()
        raise AssertionError("A autenticação aceitou token vazio.")
    except ValueError:
        pass

    import tempfile
    with tempfile.TemporaryDirectory() as pasta_temporaria:
        arquivo_mapa = Path(pasta_temporaria) / "mapa_teste.html"
        mapa_teste = gerar_mapa(
            gtfs, melhor, origem_teste, destino_teste, paradas, forma,
            caminho_html=arquivo_mapa
        )
        assert arquivo_mapa.exists() and arquivo_mapa.stat().st_size > 1000
        assert "polyline" in arquivo_mapa.read_text(encoding="utf-8").lower()

    print("Testes locais aprovados: coordenadas, distâncias, GTFS, paradas próximas, viagem direta, sequência, shape, API simulada e mapa HTML.")


executar_testes_locais()
